# 🎙️ 個人聲音克隆與文字朗讀 (Voice Clone TTS) - Colab GPU 執行手冊

本筆記本可直接於 **Google Colab (啟用免費 GPU)** 運行，並可隨時將程式碼推播至您的 GitHub Repository。

### ⚠️ 執行前第一步：確認 GPU 加速器
請點擊 Colab 上方選單：`執行階段 (Runtime)` -> `變更執行階段類型 (Change runtime type)` -> 選取 **T4 GPU** -> 儲存。

In [ ]:
# 步驟 1: 驗證 GPU 環境
!nvidia-smi

In [ ]:
# 步驟 2: 安裝高品質聲音克隆核心與 Web 工具 (以 F5-TTS 為例)
!pip install -q f5-tts gradio torchaudio soundfile pyngrok

In [ ]:
# 步驟 3: 建立聲音克隆與推論程式 (可直接由 Colab 啟動 Gradio 互動網頁)
import os
import torchaudio
from f5_tts.api import F5TTS

# 初始化預訓練模型權重 (載入至 GPU)
f5tts_engine = F5TTS(model_type="F5-TTS")
print("✅ 聲音克隆模型載入成功！")

In [ ]:
# 步驟 4: 定義克隆與文字朗讀函式
def clone_voice_and_speak(ref_audio_path, ref_transcript, target_text, output_path="output.wav"):
    """
    :param ref_audio_path: 你的 5-10 秒說話音訊路徑 (wav 或 mp3)
    :param ref_transcript: 該音檔所說的逐字文字 (提供以提升對齊準度)
    :param target_text: 你想讓自己聲音朗讀的新文本內容
    """
    wav, sr, _ = f5tts_engine.infer(
        ref_file=ref_audio_path,
        ref_text=ref_transcript,
        gen_text=target_text,
        file_wave_name=output_path
    )
    return output_path

print("函式就緒，可直接呼叫或掛載至互動介面。")

In [ ]:
# 步驟 5: 啟動互動式 Web 測試介面 (Colab 會自動產出公開可訪問連結)
import gradio as gr

def tts_interface(ref_audio, ref_text, target_text):
    if ref_audio is None:
        return None, "請先錄製或上傳您的聲音樣本！"
    
    out_file = clone_voice_and_speak(
        ref_audio_path=ref_audio,
        ref_transcript=ref_text,
        target_text=target_text
    )
    return out_file, "合成完成！"

ui = gr.Interface(
    fn=tts_interface,
    inputs=[
        gr.Audio(type="filepath", label="1. 上傳或直接麥克風錄製 5-10 秒聲音樣本"),
        gr.Textbox(label="2. 樣本語音所唸的文字 (選填但推薦)", value="這是一段我本人的測試錄音。"),
        gr.Textbox(label="3. 想要朗讀的全新文字內容", lines=4, value="歡迎來到人工智慧的世界！這是用我的聲音克隆出的朗讀效果。"),
    ],
    outputs=[
        gr.Audio(label="生成的朗讀音訊 (點擊播放/下載)"),
        gr.Textbox(label="處理狀態")
    ],
    title="🎙️ 個人聲音克隆朗讀器 (Colab 驅動版)",
    description="在 Colab 透過 GPU 運算，無需安裝本地重度環境。"
)

ui.launch(share=True)